# 02 · Dataset común 2025–2026, partición Out-of-Time y auditoría
Une el S4 reconstruido con los precursores **causalizados** (ver `s4.features`), busca el corte cronológico (por día ionosférico que empieza a las 12 UT = 07 LT) y audita la ausencia de fuga. Las fechas elegidas deben **copiarse al YAML** (`split.fin_train`, `split.fin_validacion`) para que todos los experimentos usen exactamente la misma partición.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, variante, ruta
from s4.pipeline import preparar_datos
from s4 import features as ft
cfg = cargar_config('../config/ablacion_2025_2026.yaml')

In [ ]:
datos = preparar_datos(cfg)
datos['corte']['ranking']

In [ ]:
print("COPIAR AL YAML ->", "fin_train:", datos["corte"]["fin_train"].date(), "| fin_validacion:", datos["corte"]["fin_validacion"].date())
datos["reporte"].to_csv(ruta(cfg, "reportes") / "tables/tabla_particion.csv", index=False)

In [ ]:
# Diagnóstico de la fuga en los índices interpolados (evidencia para la tesis)
from s4 import io
mv = io.cargar_multivariable(ruta(cfg, "multivariable"))
for c in ["Dst_Index", "AE_Index", "TEC", "ROTI"]:
    print(c, ft.diagnosticar_interpolacion(mv[c])["pct_minutos_lineales"], "% de minutos en tramos lineales")
dia = "2025-03-15"
fig, ax = plt.subplots(figsize=(12, 3))
mv.loc[dia, "Dst_Index"].plot(ax=ax, label="Dst en el CSV (interpolado)")
ft.causalizar_indice(mv["Dst_Index"], "1h").loc[dia].plot(ax=ax, label="Dst causal")
ax.legend(); plt.show()

In [ ]:
# Mapa temporal de la partición
df, ev = datos["df"], datos["eventos"]
diario = df["S4"].resample("1D").max()
fig, ax = plt.subplots(figsize=(14, 3))
for n, p, c in [("Train", datos["train"], "tab:blue"), ("Validación", datos["val"], "tab:orange"), ("Test", datos["test"], "tab:green")]:
    ax.axvspan(p.index.min(), p.index.max(), color=c, alpha=0.12, label=n)
ax.plot(diario.index, diario.values, color="k", lw=0.8); ax.axhline(0.6, color="r", ls=":")
ax.set_ylabel("S4 máx. diario"); ax.legend(); plt.tight_layout()
plt.savefig(ruta(cfg, "reportes") / "figures/fig_particion_oot.png", dpi=200); plt.show()